# 🧬 Cancer Prediction Using Machine Learning

## Phase 10 — Predictions on New Data

**Objective:** Build a reusable prediction pipeline that can take raw measurements
from a new patient, apply the exact same preprocessing, and output a diagnosis.

---

### 📋 Table of Contents

1. Setup & Load Model Artifacts
2. The Inference Pipeline
3. Single Patient Prediction (Code)
4. Interactive Prediction Interface (UI)
5. Phase 10 Summary

In [ ]:
# =============================================================================
# CELL 1: Setup & Load Artifacts
# =============================================================================

import os, sys, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import ipywidgets as widgets
from IPython.display import display, HTML

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import print_section_header

print_section_header("Loading ML Artifacts", "📦")

# 1. Load the trained model
model = joblib.load('../models/cancer_prediction_model.pkl')
print(f"✅ Model loaded: {type(model).__name__}")

# 2. Load the fitted scaler
scaler = joblib.load('../models/scaler.pkl')
print(f"✅ Scaler loaded: {type(scaler).__name__}")

# 3. Load the selected features list
with open('../data/processed/selected_features.json', 'r') as f:
    selected_features = json.load(f)
print(f"✅ Selected features loaded: {len(selected_features)} features")

# Load a sample of raw data to get realistic default values for our UI
raw_df = pd.read_csv('../data/raw/breast_cancer.csv')
benign_sample = raw_df[raw_df['diagnosis'] == 'B'].iloc[0]
malignant_sample = raw_df[raw_df['diagnosis'] == 'M'].iloc[0]

---

## 2. ⚙️ The Inference Pipeline

To make a prediction on a new patient, we must apply the **exact same transformations**
that were applied to the training data. If we skip scaling, the prediction will be garbage.

**Pipeline Steps:**
1. Receive raw patient data (dict or JSON)
2. Convert to DataFrame
3. Filter to keep only `selected_features`
4. Scale using the saved `scaler`
5. Predict using the saved `model`

In [ ]:
# =============================================================================
# CELL 2: Prediction Function
# =============================================================================

def predict_diagnosis(patient_data: dict) -> dict:
    """
    End-to-end inference pipeline for a single patient.
    """
    # 1. Convert dict to DataFrame (single row)
    df_patient = pd.DataFrame([patient_data])
    
    # 2. Keep only the features the model was trained on
    # Note: If a feature is missing, this will raise a KeyError (which is correct — we need all inputs)
    X_new = df_patient[selected_features]
    
    # 3. Scale the features using our PRE-FITTED scaler
    X_scaled = pd.DataFrame(
        scaler.transform(X_new),
        columns=X_new.columns
    )
    
    # 4. Make prediction
    prediction = model.predict(X_scaled)[0]
    probability = model.predict_proba(X_scaled)[0]
    
    # 5. Format results
    diagnosis = "Malignant" if prediction == 1 else "Benign"
    confidence = probability[prediction] * 100
    
    return {
        'prediction_code': int(prediction),
        'diagnosis': diagnosis,
        'confidence_pct': round(confidence, 2),
        'probabilities': {
            'Benign': round(probability[0] * 100, 2),
            'Malignant': round(probability[1] * 100, 2)
        }
    }

print("✅ Inference pipeline function defined")

In [ ]:
# =============================================================================
# CELL 3: Test with Code
# =============================================================================

print_section_header("Testing Prediction Pipeline", "🧪")

# Create a patient profile from our raw data
test_patient_m = malignant_sample[selected_features].to_dict()
test_patient_b = benign_sample[selected_features].to_dict()

print("Test Case 1 (Expected: Malignant):")
result_m = predict_diagnosis(test_patient_m)
print(f"  Result: {result_m['diagnosis']} (Confidence: {result_m['confidence_pct']}%) \n")

print("Test Case 2 (Expected: Benign):")
result_b = predict_diagnosis(test_patient_b)
print(f"  Result: {result_b['diagnosis']} (Confidence: {result_b['confidence_pct']}%) \n")

---

## 3. 🎛️ Interactive Prediction Interface

Let's build a UI inside the notebook using `ipywidgets`. This allows a doctor to
manually input the top 5 most important features and get an instant prediction.

In [ ]:
# =============================================================================
# CELL 4: Interactive UI
# =============================================================================

# Get the top 5 most important features (we'll only ask for these to keep the UI clean,
# and use median values for the rest)
top_5_features = [
    'concave points_worst',
    'perimeter_worst',
    'concave points_mean',
    'radius_worst',
    'area_worst'
]

# Calculate medians for the hidden features
medians = raw_df[selected_features].median().to_dict()

# Create UI elements
style = {'description_width': 'initial'}
sliders = {}

display(HTML("<h3>🩺 Cancer Diagnostic Tool</h3>"))
display(HTML("<p>Enter the top 5 cellular measurements to predict malignancy.</p>"))

for feature in top_5_features:
    # Safely get min, max, median for the slider
    if feature in raw_df.columns:
        f_min = float(raw_df[feature].min())
        f_max = float(raw_df[feature].max())
        f_med = float(raw_df[feature].median())
    else:
        f_min, f_max, f_med = 0.0, 100.0, 10.0 # Fallback
        
    clean_name = feature.replace('_', ' ').title()
    sliders[feature] = widgets.FloatSlider(
        value=f_med, min=f_min, max=f_max, step=(f_max-f_min)/100,
        description=f"{clean_name}:", style=style, layout=widgets.Layout(width='500px')
    )
    display(sliders[feature])

btn = widgets.Button(
    description='Predict Diagnosis',
    button_style='primary',
    icon='stethoscope',
    layout=widgets.Layout(width='200px', height='40px', margin='20px 0 0 0')
)
output = widgets.Output()

display(btn, output)

def on_click(b):
    with output:
        output.clear_output()
        
        # Start with medians for all selected features
        patient_data = medians.copy()
        
        # Update with values from sliders
        for feature in top_5_features:
            if feature in sliders:
                patient_data[feature] = sliders[feature].value
            
        # Predict
        result = predict_diagnosis(patient_data)
        
        # Display formatted result
        color = "#f85149" if result['prediction_code'] == 1 else "#3fb950"
        emoji = "🔴" if result['prediction_code'] == 1 else "🟢"
        
        html = f"""
        <div style="border: 2px solid {color}; padding: 15px; border-radius: 8px; margin-top: 15px; width: 470px;">
            <h2 style="margin: 0; color: {color};">{emoji} {result['diagnosis']}</h2>
            <p style="font-size: 16px; margin: 10px 0 0 0;">
                Confidence: <b>{result['confidence_pct']}%</b>
            </p>
            <hr style="border-color: #555; margin: 10px 0;">
            <small><i>Note: Features not shown in sliders were filled with dataset medians.</i></small>
        </div>
        """
        display(HTML(html))

btn.on_click(on_click)

---

## 4. 📋 Phase 10 Summary

### ✔ Summary

- Built a complete **Inference Pipeline** (`predict_diagnosis`)
- Handles the critical step of applying the **same scaler** used in training
- Created an **Interactive UI** using `ipywidgets` right in the notebook
- Extracted inference logic which can be moved to a standalone `src/predict.py` or Gradio app.

### ✔ What You Learned

| Concept | Description |
|---------|-------------|
| Inference Pipeline | The code that takes new data, preprocesses it, and predicts |
| Scaler Preservation | Why you must load the fitted scaler from training |
| ipywidgets | Building interactive interfaces in Jupyter |

**🎯 Interview Question:** *If you train a model with scaled data, what happens if you forget to scale a new patient's data before predicting?*
> The prediction will be completely wrong. The model learned coefficients based on scaled data (mean=0, std=1). If you pass raw data (e.g., area=1000), the model will treat it as a massive outlier (1000 standard deviations above mean), resulting in an extreme and incorrect prediction.

---

*Phase 10 Complete ✅*